In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
# data <- data[!is.na(data$Atyp_10pct_Z), ]
# dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2768162      78

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2768162      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total <- feols(fml, data = data, vcov = "iid")
summary(model_total)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: author_id: 78,875,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.002974   0.000080  37.064165
num_author_log                                0.000054   0.000061   0.884001
num_inst_log                                  0.001211   0.000078  15.545211
num_country_log                              -0.002973   0.000115 -25.961477
num_reference_log                             0.002141   0.000039  54.356877
leadership_global_classallNorth               0.001486   0.000117  12.668260
leadership_global_classallSouth               0.000807   0.000184   4.399553
mean_career_age_log                           0.000559   0.000065   8.616935
inst_h_index_log                              0.000351   0.000050   7.079320
source_h_index_log                           -0.000787   0.000029 -27.011958
core_sourceCore        

In [13]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

# disciplines <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
#                  "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
#                  "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
#                  "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
#                  "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
#                  "Psychology", "Social.Sciences", "Veterinary")

In [14]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("fac_pub", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & rao\_stirling\\   
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.003$^{***}$\\   
                                       & ($8.02\times 10^{-5}$)\\    
   num\_author\_log                    & $5.38\times 10^{-5}$\\    
                                       & ($6.08\times 10^{-5}$)\\    
   num\_inst\_log                      & 0.001$^{***}$\\   
                                       & ($7.79\times 10^{-5}$)\\    
   num\_country\_log                   & -0.003$^{***}$\\   
                                       & (0.0001)\\   
   num\_reference\_log                 & 0.002$^{***}$\\   
                                       & ($3.94\times 10^{-5}$)\\    
   leadership\_global\_classallNorth   & 0.001$^{***}$\\   
                                       & (0.0001)\\   
   leadership\_glo

In [15]:
# 每组 reg_class 的平均预测概率
margins_eff_reg_class <- avg_predictions(model_total, variables = "fac_pub")
margins_eff_reg_class

fac_pub,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonBSF,0.1375701,0.0004315063,318.8136,0,Inf,0.1367243,0.1384158
BSF,0.1405444,0.0004311777,325.9547,0,Inf,0.1396993,0.1413895


In [16]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211.csv")
write.csv(margins_eff_reg_class, fname, row.names = FALSE)

In [13]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", disciplines, " | author_id + year")
)

model_total1 <- feols(fml, data = data, vcov = "iid")
summary(model_total1)

NOTE: 1 observation removed because of NA values (LHS: 1).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,161
Fixed-effects: author_id: 78,875,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   t value
fac_pubBSF                                    0.003056   0.000078  38.93840
Arts.and.Humanities                           0.044948   0.000395 113.73808
Biochemistry..Genetics.and.Molecular.Biology  0.016119   0.000077 210.27224
Business..Management.and.Accounting           0.103352   0.000465 222.15428
Chemical.Engineering                         -0.002991   0.000109 -27.40948
Chemistry                                     0.006452   0.000067  96.32355
Computer.Science                              0.034304   0.000155 221.84165
Decision.Sciences                             0.032081   0.000428  75.03925
Dentistry                                     0.044282   0.000421 105.11500
Earth.and.Planetary.Sciences                  0.008905   0.000107  83.50573
Economics..Econometrics.and.Financ

In [14]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", disciplines, " | author_id + year")
)

model_total2 <- feols(fml, data = data, vcov = "iid")
summary(model_total2)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: author_id: 78,875,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   t value
fac_pubBSF                                    0.002864   0.000080  35.72685
num_author_log                               -0.000132   0.000060  -2.17715
num_inst_log                                  0.001181   0.000078  15.16144
num_country_log                              -0.002946   0.000115 -25.72657
num_reference_log                             0.001966   0.000039  50.58635
leadership_global_classallNorth               0.001439   0.000117  12.26947
leadership_global_classallSouth               0.000848   0.000184   4.62023
mean_career_age_log                           0.000555   0.000065   8.55183
inst_h_index_log                              0.000278   0.000049   5.61320
Arts.and.Humanities                           0.045242   0.000395 114.53309
Biochemistry..Genetics.and.Molecul

In [15]:
fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total3 <- feols(fml, data = data, vcov = "iid")
summary(model_total3)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: author_id: 78,875,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.002974   0.000080  37.064165
num_author_log                                0.000054   0.000061   0.884001
num_inst_log                                  0.001211   0.000078  15.545211
num_country_log                              -0.002973   0.000115 -25.961477
num_reference_log                             0.002141   0.000039  54.356877
leadership_global_classallNorth               0.001486   0.000117  12.668260
leadership_global_classallSouth               0.000807   0.000184   4.399553
mean_career_age_log                           0.000559   0.000065   8.616935
inst_h_index_log                              0.000351   0.000050   7.079320
source_h_index_log                           -0.000787   0.000029 -27.011958
core_sourceCore        

In [16]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines_vars <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [17]:
model1 <- model_total1
model2 <- model_total2
model3 <- model_total3

country_level_reg <- etable(
    list(model1, model2, model3),
    keep = c("fac_pub", paper_vars, journal_vars, disciplines_vars),
    se = "iid",
    # cluster = ~ paper_year + paper_field,
    tex = TRUE,
    digits = 3
)
country_level_reg
# writeLines(country_level_reg, con = "selfpromt_reg_1014.tex")

\begingroup
\centering
\begin{tabular}{lccc}
   \tabularnewline \midrule \midrule
   Dependent Variable: & \multicolumn{3}{c}{rao\_stirling}\\
   Model:                                       & (1)                     & (2)                     & (3)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                                  & 0.003$^{***}$           & 0.003$^{***}$           & 0.003$^{***}$\\   
                                                & ($7.85\times 10^{-5}$)  & ($8.02\times 10^{-5}$)  & ($8.02\times 10^{-5}$)\\    
   Arts.and.Humanities                          & 0.045$^{***}$           & 0.045$^{***}$           & 0.045$^{***}$\\   
                                                & (0.0004)                & (0.0004)                & (0.0004)\\   
   Biochemistry..Genetics.and.Molecular.Biology & 0.016$^{***}$           & 0.016$^{***}$           & 0.016$^{***}$\\   
                                                & ($7.67\times 10^{-5}$)  & ($7.66\times 10^{-5}$)  & (